### 3.6.5. Mixed-Integer Nonlinear Programming (MINLP)

$$
\begin{aligned}
\min_{\mathbf{x}}\quad & f(\mathbf{x}) \\
\text{subject to}\quad & g_i(\mathbf{x}) \le 0,\quad i=1,\dots,m,\\
& x_j \in \mathbb{Z}\ \text{for } j\in\mathcal{I},
\end{aligned}
$$
with $f, g_i$ nonlinear.

**Explanation:**

A mixed-integer nonlinear program (MINLP) combines the two hardest ingredients: nonlinear (possibly nonconvex) functions and integrality of some variables. It generalizes every earlier class in this module. Solution methods interleave a continuous [nonlinear program](../05_Nonlinear_Programming/01_general_nonlinear_programs.ipynb) relaxation with branching on the integer variables; solvers such as Bonmin implement this branch-and-bound-plus-NLP strategy. When the continuous relaxation is convex the bounds are valid and the method is exact.

**Numerical Example:**

$$
\min\; e^{-x_1} + (x_2 - 2)^2
\quad\text{s.t.}\quad x_1 + x_2 \le 3,\; x_1\in\{0,1,2,3\},\; x_2\in\mathbb{R}_{\ge 0}.
$$

For each integer $x_1$ the best continuous $x_2$ is $\min(2,\,3-x_1)$ (the unconstrained optimum $2$, clamped by the constraint):
$$
x_1=0:\ x_2=2,\ e^{0}=1.000;\quad
x_1=1:\ x_2=2,\ e^{-1}=0.368;\quad
x_1=2:\ x_2=1,\ e^{-2}+1=1.135.
$$
The optimum is $x_1=1,\ x_2=2$ with value $\approx 0.368$, a genuinely nonlinear trade-off no linear model would capture.

In [1]:
import sympy as sp

x_1_choices = range(0, 4)
results = {}
for x_1 in x_1_choices:
    x_2 = min(2, 3 - x_1)
    if x_2 >= 0:
        results[(x_1, x_2)] = float(sp.exp(-x_1) + (x_2 - 2)**2)

best_point = min(results, key=results.get)

print("objective per integer choice:", {k: round(v, 3) for k, v in results.items()})
print("optimum:", best_point, "value:", round(results[best_point], 3))

objective per integer choice: {(0, 2): 1.0, (1, 2): 0.368, (2, 1): 1.135, (3, 0): 4.05}
optimum: (1, 2) value: 0.368


**Equivalent casadi (bonmin) implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
objective = ca.exp(-decision_variable[0]) + (decision_variable[1] - 2)**2
constraint = decision_variable[0] + decision_variable[1]

solver = ca.nlpsol("solver", "bonmin",
                   {"x": decision_variable, "f": objective, "g": constraint},
                   {"discrete": [True, False], "bonmin.bb_log_level": 0, "print_time": 0})
solution = solver(x0=[0, 0], lbg=-ca.inf, ubg=3, lbx=0, ubx=[3, ca.inf])

print("optimum:", solution["x"], " value:", round(float(solution["f"]), 3))


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT 0.339077        7 0.001222
NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT 0.36787944       13 0.001606
NLP0014I             2         OPT 0.36787945       12 0.001541
NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT 0.36787945       10 0.001391
NLP0014I             2         OPT 0.36787944       13 0.00

**References:**

[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)  
[📗 Postek, K., & Zocca, A. (2024). *Hands-On Mathematical Optimization with Python*, Ch. 3–4. Cambridge University Press.](https://mobook.github.io/MO-book/intro.html)

---

[⬅️ Previous: Mixed-Integer Quadratic Programming (MIQP)](./04_mixed_integer_quadratic_programming.ipynb) | [Next: Combinatorial Optimization ➡️](./06_combinatorial_optimization.ipynb)